# 04 · Entropy, cross-entropy, KL and perplexity

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/math/04_cross_entropy_kl.ipynb) &nbsp; 📖 Concept deck: [Information theory for LLMs](https://nilesh-salpe.github.io/visual-ai/math/07-information-theory/) &nbsp; ⏱️ Suggested time: **25 min**

**You will build:** entropy, cross-entropy and KL divergence from scratch, the identity H(p, q) = H(p) + KL(p‖q), and perplexity of a language model from its per-token probabilities.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")

## Task 1: Entropy in nats and bits

$H(p) = -\sum_i p_i \log p_i$, with the convention $0 \log 0 = 0$. Return `(nats, bits)`; bits use $\log_2$.

In [ ]:
def entropy(p):
    p = np.asarray(p, float)
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_entropy(f):
    assert np.isclose(f([0.5, 0.5])[1], 1), "a fair coin is 1 bit"
    assert np.isclose(f([1, 0, 0])[0], 0), "a certain outcome has zero entropy (watch 0·log 0)"
    assert np.isclose(f(np.ones(8) / 8)[1], 3), "uniform over 8 is 3 bits"
    n, b = f([0.7, 0.2, 0.1]); assert np.isclose(n, b * np.log(2))

check("entropy", _test_entropy, entropy)

## Task 2: Cross-entropy and KL divergence

Return `(H_pq, KL_pq)` with $H(p, q) = -\sum p_i \log q_i$ and $\mathrm{KL}(p \Vert q) = \sum p_i \log (p_i / q_i)$, in nats.
Training a language model minimises cross-entropy against the data distribution p; since H(p) is fixed, that is the same as minimising KL(p‖q).

In [ ]:
def cross_entropy_kl(p, q):
    p, q = np.asarray(p, float), np.asarray(q, float)
    # TODO (terms with p_i = 0 contribute 0)
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_cross_entropy_kl(f):
    p, q = np.array([0.7, 0.2, 0.1]), np.array([0.5, 0.3, 0.2])
    H, KL = f(p, q)
    Hp = -np.sum(p * np.log(p))
    assert np.isclose(H, Hp + KL), "H(p,q) = H(p) + KL(p||q)"
    assert KL > 0 and np.isclose(f(p, p)[1], 0), "KL ≥ 0, with equality iff q = p"
    assert not np.isclose(f(p, q)[1], f(q, p)[1]), "KL is not symmetric"
    assert np.isclose(f([1, 0], [0.5, 0.5])[0], np.log(2))

check("cross_entropy_kl", _test_cross_entropy_kl, cross_entropy_kl)

## Task 3: Perplexity from token probabilities

Given the probabilities a model assigned to each **actual** next token in a text, return `(loss, ppl)`: the mean negative log-likelihood in nats and the perplexity $e^{\text{loss}}$.
Perplexity is the size of a uniform choice that would be equally surprising.

In [ ]:
def perplexity(token_probs):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_perplexity(f):
    loss, ppl = f([0.25] * 10)
    assert np.isclose(ppl, 4), "always giving the right token probability 1/4 = perplexity 4"
    assert np.isclose(f([1 / 50257] * 3)[0], np.log(50257)), "uniform over GPT-2's 50,257 tokens = ln 50257 nats"
    l, p = f([0.9, 0.5, 0.01]); assert np.isclose(p, np.exp(l))

check("perplexity", _test_perplexity, perplexity)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Entropy in nats and bits
def entropy_solution(p):
    p = np.asarray(p, float); nz = p[p > 0]
    nats = -np.sum(nz * np.log(nz))
    return nats, nats / np.log(2)

check("entropy (reference)", _test_entropy, entropy_solution)

In [ ]:
# @title Solution for Task 2: Cross-entropy and KL divergence
def cross_entropy_kl_solution(p, q):
    p, q = np.asarray(p, float), np.asarray(q, float); m = p > 0
    return -np.sum(p[m] * np.log(q[m])), np.sum(p[m] * np.log(p[m] / q[m]))

check("cross_entropy_kl (reference)", _test_cross_entropy_kl, cross_entropy_kl_solution)

In [ ]:
# @title Solution for Task 3: Perplexity from token probabilities
def perplexity_solution(token_probs):
    loss = -np.mean(np.log(np.asarray(token_probs, float)))
    return loss, np.exp(loss)

check("perplexity (reference)", _test_perplexity, perplexity_solution)

---
## Experiments
**Forward vs reverse KL.** Fit a single Gaussian-like bump q to a two-peaked p over a grid. Minimising KL(p‖q) spreads q to cover both peaks; minimising KL(q‖p) locks onto one. Which one does maximum likelihood use?

In [ ]:
x = np.linspace(-6, 6, 241)
p = np.exp(-(x + 2.5) ** 2 / 0.5) + np.exp(-(x - 2.5) ** 2 / 0.5); p /= p.sum()
best = {}
for mu in np.linspace(-4, 4, 81):
    for s in [0.5, 1, 2, 3]:
        q = np.exp(-(x - mu) ** 2 / (2 * s * s)); q /= q.sum(); q = np.maximum(q, 1e-300)
        fwd, rev = np.sum(p * np.log(p / q + 1e-300)), np.sum(q * np.log(q / np.maximum(p, 1e-300)))
        for k, v in (('forward KL(p||q)', fwd), ('reverse KL(q||p)', rev)):
            if k not in best or v < best[k][0]: best[k] = (v, mu, s)
for k, (v, mu, s) in best.items(): print(f"{k}: best q has mean {mu:+.1f}, width {s}")

---
## Takeaways
* Entropy is average surprise; cross-entropy is surprise under the wrong model; KL is the gap between them.
* LM training loss is cross-entropy in nats per token; perplexity = e^loss.
* Maximum likelihood minimises forward KL(p‖q), which is mass-covering.